# Email Attack Detection Using Ensemble Machine Learning Techniques

This notebook implements the original project workflow for detecting phishing emails from email text using TF-IDF, feature scaling, PCA, Logistic Regression, Decision Tree, and a Stacking Classifier.

**Dataset file expected:** `data/Phishing_Email.csv`

> Note: The preprocessing order and reported results are preserved from the original project notebook. See the README for an important reproducibility note about preprocessing and data leakage.

In [ ]:
# Import required libraries
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

# Load dataset from the repository's data directory.
DATA_PATH = os.path.join("data", "Phishing_Email.csv")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Download the dataset and place Phishing_Email.csv in the data/ folder. "
        "See data/README.md for details."
    )

data = pd.read_csv(DATA_PATH)

# Remove an exported index column if present.
data.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)

# Focus on relevant columns.
data = data[["Email Text", "Email Type"]]
data["Email Text"] = data["Email Text"].fillna("")

# Encode target labels.
label_encoder = LabelEncoder()
data["Email Type"] = label_encoder.fit_transform(data["Email Type"])

print(f"Dataset shape: {data.shape}")
print(f"Label distribution:\n{data['Email Type'].value_counts()}")
print(data.info())
print(data.head())


In [ ]:
# TF-IDF vectorization, scaling, and PCA
tfidf = TfidfVectorizer(max_features=700)
X_text = tfidf.fit_transform(data["Email Text"]).toarray()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_text)

# Reduce dimensionality while retaining 90% of variance.
pca = PCA(n_components=0.90)
X_reduced = pca.fit_transform(X_scaled)

print(f"Original features: {X_text.shape[1]}, Reduced features: {X_reduced.shape[1]}")

# Split data into training and testing sets.
y = data["Email Type"]

X_train, X_test, y_train, y_test = train_test_split(
    X_reduced,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Testing set shape: {X_test.shape}, {y_test.shape}")


In [ ]:
# Base model training and evaluation
logreg = LogisticRegression(random_state=42, max_iter=1000)
tree = DecisionTreeClassifier(random_state=42)

logreg.fit(X_train, y_train)
tree.fit(X_train, y_train)

logreg_pred = logreg.predict(X_test)
tree_pred = tree.predict(X_test)

def evaluate_model(name, y_true, y_pred):
    precision = precision_score(y_true, y_pred, average="weighted")
    recall = recall_score(y_true, y_pred, average="weighted")
    f1 = f1_score(y_true, y_pred, average="weighted")
    print(
        f"{name} - Precision: {precision:.4f}, "
        f"Recall: {recall:.4f}, F1 Score: {f1:.4f}"
    )
    return precision, recall, f1

evaluate_model("Logistic Regression", y_test, logreg_pred)
evaluate_model("Decision Tree", y_test, tree_pred)


In [ ]:
# Stacking ensemble
stacking_clf = StackingClassifier(
    estimators=[
        ("LogReg", logreg),
        ("Tree", tree),
    ],
    final_estimator=LogisticRegression(random_state=42, max_iter=1000),
    cv=5,
)

stacking_clf.fit(X_train, y_train)
stacking_pred = stacking_clf.predict(X_test)

evaluate_model("Stacking Classifier", y_test, stacking_pred)


In [ ]:
# Confusion matrix and classification report
cm = confusion_matrix(y_test, stacking_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_,
)
disp.plot()

print("Classification Report:")
print(
    classification_report(
        y_test,
        stacking_pred,
        target_names=label_encoder.classes_,
    )
)


In [ ]:
# Cross-validation and model export
cv_scores = cross_val_score(
    stacking_clf,
    X_reduced,
    y,
    cv=5,
    scoring="f1_weighted",
)

print(f"Cross-validation F1 Scores: {cv_scores}")
print(f"Mean CV F1 Score: {cv_scores.mean():.4f}")

# Inspect the test-set feature vectors corresponding to misclassified samples.
misclassified = X_test[y_test != stacking_pred]
print(f"Number of misclassified test samples: {len(misclassified)}")

# Save trained components for later experimentation/deployment.
joblib.dump(stacking_clf, "stacking_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
joblib.dump(pca, "pca_pipeline.pkl")
